# 01 — Rebuilding the book, and the one axis the panel varies along

Nasdaq does not publish the order book.  It publishes the *changes* to it, one
order at a time, as a binary feed: an add, an execution, a cancel, a delete, a
replace, each carrying an order reference and a nanosecond timestamp.  This
notebook checks that we can turn that back into a book, and then measures the
variable everything else in the repository is organised around: how many ticks
wide the spread is.

The tick on Nasdaq is one cent for every stock above a dollar.  That makes the
*price level* the tick axis: a cent is 17 basis points of SIRI at \$6 and 0.06
basis points of AMZN at \$1650, and the two markets that result behave
differently in almost every measurement that follows.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hfx.viz import subplots, use_style, colour_for, tick_regime
from hfx.pipeline.results import panel, curves, curve, symbol_days
use_style()
P = panel(); C = curves()
DAY = "2019-01-30"
print(f"{len(P)} symbol-days, {P.symbol.nunique()} symbols, {P.date.nunique()} dates")

## The wire format, on a book small enough to check by hand

`hfx.itch.synth` writes ITCH messages from the published field offsets; the
decoder reads them back.  The two were written from the specification
independently, so this is a real check of the binary layout rather than a
round-trip through one piece of code.

In [ ]:
from hfx.itch import synth
from hfx.itch.reader import ItchExtractor
from hfx.itch.spec import OPEN_NS

S = 1_000_000_000
messages = [
    synth.stock_directory(1, "DEMO"),
    synth.add_order(1, OPEN_NS + 1 * S, ref=10, side=+1, shares=300, price=1_500_000),
    synth.add_order(1, OPEN_NS + 2 * S, ref=11, side=-1, shares=200, price=1_500_100),
    synth.execute(1, OPEN_NS + 3 * S, ref=11, shares=120, match=1),
    synth.cancel(1, OPEN_NS + 4 * S, ref=10, shares=100),
]
ext = ItchExtractor(["DEMO"])
ext.feed(synth.frame(messages))
decoded = pd.DataFrame(ext.buffers["DEMO"].as_dict())
decoded["type"] = [chr(c) for c in decoded.etype]
decoded[["type", "ts", "ref", "side", "shares", "price"]]

## Replay: the book, the trades, and the aggressor side

An execution consumes a *resting* order whose side the book knows, so the
aggressor is its opposite.  No Lee-Ready inference, no tick test, and therefore
none of the attenuation a sign-inference error inflicts on every impact
estimate downstream.

In [ ]:
from hfx.book.replay import replay

out = replay({k: np.asarray(v) for k, v in ext.buffers["DEMO"].as_dict().items()}, "DEMO", DAY)
trade = {k: v[0] for k, v in out.trades.items()}
print("one trade:", {k: int(v) for k, v in trade.items()})
assert trade["side"] == +1, "an execution against a resting sell is a buy"
assert trade["price"] == 1_500_100 and trade["size"] == 120
assert out.stats["n_unknown_ref"] == 0

## The panel

Twelve Nasdaq-listed symbols over seven days, chosen to run from \$6 to
\$1 700 so that the spread-to-tick ratio runs from one to fifty.

In [ ]:
day = P[P.date == DAY].sort_values("price")
cols = ["symbol", "price", "median_spread_ticks", "p_spread_one_tick", "n_trades",
        "trades_per_second", "mean_order_size", "aes", "hidden_share", "at_touch_displayed"]
day[cols].round(3).to_string(index=False)

In [ ]:
fig, axes = subplots(1, 2, figsize=(11, 3.6))
g = P.groupby("symbol").agg(price=("price", "mean"),
                            p1=("p_spread_one_tick", "mean"),
                            spread=("median_spread_ticks", "mean"),
                            queue=("qr_mean_queue_emp", "mean")).sort_values("price")
for ax, (col, label) in zip(axes, [("p1", "time with a one-tick spread"),
                                   ("queue", "best queue, in average event sizes")]):
    for sym, r in g.iterrows():
        ax.scatter(r.price, r[col], color=colour_for(r.spread), s=45)
        ax.annotate(sym, (r.price, r[col]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("price ($)"); ax.set_ylabel(label)
axes[0].set_title("Large tick at the left, small tick at the right")
axes[1].set_title("Queues exist only where the tick binds")
plt.tight_layout()

The two panels are the same statement twice.  A stock whose tick is
large relative to its volatility cannot have a spread wider than one tick for
long, so liquidity has nowhere to go but *into the queue at the touch*; a stock
whose tick is negligible has a spread of tens of ticks and queues barely one
order deep.  Every later chapter splits along this line.

In [ ]:
fig, axes = subplots(1, 2, figsize=(11, 3.4))
for sym, ax in zip(["SIRI", "AMZN"], axes):
    counts = curve(C, sym, DAY, "minute_counts")
    minutes = np.arange(counts.shape[0])
    ax.plot(minutes, counts[:, 0], label="limit orders")
    ax.plot(minutes, counts[:, 1], label="cancels")
    ax.plot(minutes, counts[:, 2], label="trades")
    ax.set_yscale("log"); ax.set_title(sym)
    ax.set_xlabel("minutes after 09:30"); ax.set_ylabel("events per minute")
axes[0].legend()
plt.tight_layout()

## Does the aggressor-side convention survive contact with the data?

A trade classified as buyer-initiated has to print at or above the offer.  The
check is run on *displayed* trades; hidden executions are non-displayed
liquidity and print inside the spread by construction, which is itself the
control.

In [ ]:
chk = P.groupby("symbol").agg(at_touch=("at_touch_displayed", "mean"),
                              hidden=("hidden_share", "mean"),
                              spread=("median_spread_ticks", "mean")).sort_values("spread")
print(chk.round(4).to_string())
assert chk.at_touch.min() > 0.95, "the aggressor side or the quote alignment is wrong"
print("\nworst symbol:", chk.at_touch.idxmin(), round(chk.at_touch.min(), 4))

In [ ]:
fig, ax = subplots(figsize=(6.2, 3.4))
for sym, r in g.iterrows():
    row = P[P.symbol == sym]
    ax.scatter(r.spread, row.effective_half_spread_ticks.mean() / row.quoted_half_spread_ticks.mean(),
               color=colour_for(r.spread), s=45)
    ax.annotate(sym, (r.spread, row.effective_half_spread_ticks.mean() / row.quoted_half_spread_ticks.mean()),
                fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xscale("log"); ax.axhline(1.0, color="0.6", lw=1, ls="--")
ax.set_xlabel("median spread (ticks)"); ax.set_ylabel("effective / quoted half-spread")
ax.set_title("What a taker pays, against what the screen showed")
plt.tight_layout()

The effective half-spread is below the quoted one everywhere, and the
gap is largest for the small-tick names: with a fifty-tick spread there is room
for hidden and price-improving liquidity to sit inside it, and takers find it.
Chapter 03 shows the same number arrived at from an entirely different
direction, through the tick grid.